# City of LA — Data Cleaning & Reference Preparation

This notebook cleans the two reference CSV files and outputs them as submission-ready spreadsheets.

**Inputs:**
- `Additional data/job_titles .csv` — 668 City of LA job class titles (no header, single column)
- `Additional data/kaggle_data_dictionary - output_fields.csv` — 21-field output schema definition

**Outputs:**
- `job_titles_clean.csv`
- `data_dictionary_clean.csv`

In [2]:
import pandas as pd
import os

# os.getcwd() returns the notebook's directory when opened in Jupyter
DATA_DIR = os.path.join(os.getcwd(), "Additional data")
OUT_DIR  = os.getcwd()

print("Data dir:", DATA_DIR)
print("Output dir:", OUT_DIR)

Data dir: /Users/songchen/Downloads/cityofla/CityofLA/Additional data
Output dir: /Users/songchen/Downloads/cityofla/CityofLA


## 1. Clean `job_titles .csv`

In [3]:
# Load — no header row, single column of job titles
jt = pd.read_csv(
    os.path.join(DATA_DIR, "job_titles .csv"),
    header=None,
    names=["JOB_CLASS_TITLE"],
    dtype=str,
)

print(f"Loaded {len(jt)} rows")
jt.head()

Loaded 668 rows


,JOB_CLASS_TITLE
0,311 DIRECTOR
1,ACCOUNTANT
2,ACCOUNTING CLERK
3,ACCOUNTING RECORDS SUPERVISOR
4,ADMINISTRATIVE ANALYST


In [4]:
# --- Cleaning ---

# 1. Strip leading/trailing whitespace
jt["JOB_CLASS_TITLE"] = jt["JOB_CLASS_TITLE"].str.strip()

# 2. Drop any fully blank rows
jt = jt[jt["JOB_CLASS_TITLE"].notna() & (jt["JOB_CLASS_TITLE"] != "")]

# 3. Remove duplicates (keep first occurrence, preserve alphabetical order)
before = len(jt)
jt = jt.drop_duplicates(subset="JOB_CLASS_TITLE")
print(f"Duplicates removed: {before - len(jt)}")

# 4. Reset index
jt = jt.reset_index(drop=True)

# 5. Flag the numeric-prefix title for awareness
numeric_prefix = jt[jt["JOB_CLASS_TITLE"].str.match(r"^\d")]
if not numeric_prefix.empty:
    print(f"\nTitles with numeric prefix (noted, kept as-is):")
    print(numeric_prefix["JOB_CLASS_TITLE"].tolist())

print(f"\nFinal row count: {len(jt)}")
jt.head(10)

Duplicates removed: 2

Titles with numeric prefix (noted, kept as-is):
['311 DIRECTOR']

Final row count: 666


,JOB_CLASS_TITLE
0,311 DIRECTOR
1,ACCOUNTANT
2,ACCOUNTING CLERK
3,ACCOUNTING RECORDS SUPERVISOR
4,ADMINISTRATIVE ANALYST
5,ADMINISTRATIVE CLERK
6,ADMINISTRATIVE HEARING EXAMINER
7,ADVANCE PRACTICE PROVIDER CORRECTIONAL CARE
8,AIR CONDITIONING MECHANIC
9,AIR CONDITIONING MECHANIC SUPERVISOR


In [ ]:
# Save
jt_out = os.path.join(OUT_DIR, "job_titles_clean.csv")
jt.to_csv(jt_out, index=False)
print(f"Saved → {jt_out}")

## 2. Clean `kaggle_data_dictionary - output_fields.csv`

In [ ]:
# Load — pandas handles RFC-compliant multiline quoted fields automatically
dd = pd.read_csv(
    os.path.join(DATA_DIR, "kaggle_data_dictionary - output_fields.csv"),
    dtype=str,
)

print(f"Loaded {len(dd)} rows × {len(dd.columns)} columns")
print("Columns:", dd.columns.tolist())
dd

In [ ]:
# --- Cleaning ---

text_cols = ["Description", "Allowable Values", "Additional Notes"]

# 1. Strip leading/trailing whitespace from all string columns
for col in dd.columns:
    dd[col] = dd[col].str.strip()

# 2. Collapse embedded newlines inside text fields to a single space
for col in text_cols:
    dd[col] = dd[col].str.replace(r"\s*\n\s*", " ", regex=True).str.strip()

# 3. Fix the "in in" typo in the JOB_CLASS_TITLE description
dd["Description"] = dd["Description"].str.replace(
    "matching in in supplied", "matching in supplied", regex=False
)

# 4. Normalize "Accepts Null Values?" to lowercase yes / no
dd["Accepts Null Values?"] = dd["Accepts Null Values?"].str.strip().str.lower()

print("Typo check — should show 'matching in supplied':")
print(dd.loc[dd["Field Name"] == "JOB_CLASS_TITLE", "Description"].values)

dd

In [ ]:
# Save
dd_out = os.path.join(OUT_DIR, "data_dictionary_clean.csv")
dd.to_csv(dd_out, index=False)
print(f"Saved → {dd_out}")